# Recommandation d'artistes avec K-Means

Ce notebook montre uniquement l'utilisation du pipeline défini dans `src/recommend.py`. Chaque résultat est rattaché à la version exacte du dataset nettoyé.

In [ ]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Impossible de trouver la racine MusicRecoSystem")

sys.path.insert(0, str(PROJECT_ROOT))

import src.recommend as recommend_module
importlib.reload(recommend_module)
from src.recommend import select_and_fit_best_kmeans

## 1. Charger une version précise des données

La colonne `dataset_version` permet de vérifier que deux expériences utilisent exactement le même nettoyage.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"
listens = pd.read_parquet(DATA_PATH)

dataset_versions = listens["dataset_version"].unique()
if len(dataset_versions) != 1:
    raise ValueError("Le fichier mélange plusieurs versions de données")

pd.Series({
    "dataset_version": dataset_versions[0],
    "interactions": len(listens),
    "users": listens["user"].nunique(),
    "artist_identities": listens["artist_uid"].nunique(),
    "mbid_coverage": listens["artist_mbid"].notna().mean(),
})

## 2. Trouver le meilleur k et entraîner la solution finale

Cette cellule réalise la séparation chronologique 80/10/10, sélectionne `k` sur la validation, réentraîne sur train + validation et évalue une seule fois sur le test.

In [ ]:
import importlib
import src.recommend as recommend_module
recommend_module = importlib.reload(recommend_module)

experiment = recommend_module.select_and_fit_best_kmeans(
    listens,
    k_values=[2, 3, 4, 5, 6, 8, 10, 12, 15, 20],
    top_n=10,
    random_state=42,
    verbose=True,
)

display(experiment.tuning_results)

metrics = pd.DataFrame({
    "validation": experiment.validation_metrics,
    "test_final": experiment.test_metrics,
})

experiment_version = getattr(
    experiment,
    "dataset_version",
    listens["dataset_version"].iloc[0],
)
print(f"Dataset : {experiment_version}")
print(f"Meilleur k : {experiment.best_k}")
display(metrics)

## 3. Inspecter le modèle final

In [ ]:
print("Tailles des parties :", experiment.split_sizes)
print("Utilisateurs par cluster :")
experiment.recommender.cluster_sizes()

## 4. Obtenir 10 recommandations

Les artistes déjà écoutés sont automatiquement exclus.

In [ ]:
USER = "BKILPATRICK11"

experiment.recommender.recommend(USER, top_n=10)